In [1]:
# Code from Milestone 2 to carry over
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


DATA_PATH = Path("PL_team_matches.csv")
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df_home = df[df["HomeAway"] == "Home"].copy()


df_home = df_home.sort_values(["Team", "Date"]).reset_index(drop=True)

def add_last5_home(g):
    g = g.sort_values("Date").copy()
    for stat in ["Goals", "HS", "HST"]:
        g[f"{stat}_avg5"] = g[stat].shift(1).rolling(5, min_periods=1).mean()
    return g

df_home = (
    df_home.groupby("Team", group_keys=False)
           .apply(add_last5_home)
           .reset_index(drop=True))

df_home["Target"] = (df_home["Goals"] >= 2).astype(int)

model_cols = ["MatchID", "Date", "Time", "Team", "Opponent", "HomeAway",
              "Target", "Goals_avg5", "HS_avg5", "HST_avg5"]
df_model = df_home[model_cols].copy()


train_end = "2024-08-01"
val_end = "2025-08-01"

train = df_model[df_model["Date"] < train_end]
validation = df_model[(df_model["Date"] >= train_end) & (df_model["Date"] < val_end)]
test = df_model[df_model["Date"] >= val_end]

feature_cols = ["Goals_avg5", "HS_avg5", "HST_avg5"]
X_train, y_train = train[feature_cols], train["Target"]
X_val,   y_val   = validation[feature_cols], validation["Target"]
X_test,  y_test  = test[feature_cols], test["Target"]

In [9]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {"Dummy": make_pipeline(SimpleImputer(strategy="median"), DummyClassifier(strategy="most_frequent")),
    
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)),
    
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
                     RandomForestClassifier(n_estimators=500, max_features="sqrt", min_samples_leaf=5,
                     oob_score=True, n_jobs=-1, random_state=42)),
    
    "k-NN": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), 
            KNeighborsClassifier(n_neighbors=7, weights="distance")),
          
    "Gradient Boosting": make_pipeline(SimpleImputer(strategy="median"),
                         GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=2,
                         subsample=0.8, random_state=42))}

In [11]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate


train_sorted = train.sort_values(["Date", "Time"])
X_cv = train_sorted[feature_cols]
y_cv = train_sorted["Target"]

cv = TimeSeriesSplit(n_splits=5)

rows = []
for name, model in models.items():
    results = cross_validate(
        model, X_cv, y_cv, cv=cv,
        scoring={"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"},
        return_train_score=True)
    
    rows.append({"Model": name,
        "Accuracy": results["test_accuracy"].mean(),
        "F1": results["test_f1"].mean(),
        "ROC-AUC mean": results["test_roc_auc"].mean(),
        "ROC-AUC std": results["test_roc_auc"].std(),
        "Fit time (s)": results["fit_time"].mean(),
        "Predict time (s)": results["score_time"].mean(),})

cv_results = pd.DataFrame(rows).set_index("Model").round(3)
cv_results

,Accuracy,F1,ROC-AUC mean,ROC-AUC std,Fit time (s),Predict time (s)
Model,,,,,,
Dummy,0.519,0.000,0.500,0.000,0.008,0.019
Logistic Regression,0.593,0.446,0.601,0.030,0.041,0.022
Random Forest,0.576,0.473,0.583,0.050,2.547,0.453
k-NN,0.534,0.472,0.540,0.045,0.013,0.021
Gradient Boosting,0.562,0.474,0.569,0.039,0.508,0.018


In [12]:
# Using time-ordered cross-validation on the 2021/22-2023/24 seasons, all four learned models beat the dummy baseline
# (ROC-AUC 0.500) but only by small margins. Logistic regression performed best (ROC-AUC 0.601, accuracy 0.593)
# and was the most stable across folds (std 0.030). Random forest (0.583) and gradient boosting (0.569) came close, 
# while k-NN (0.540) was barely above the baseline. The more complex models did not beat the simple one, 
# and the random forest was far slower to fit (2.5s vs 0.04s).
#Logistic regression is the current preferred model, as it performs at least as well as the alternatives and is faster 
# and easier to interpret. A likely reason for the small gains is that the three features only describe the home team’s 
# own attack. Next steps are to compare the models on the validation season, tune lightly (e.g. k-NN neighbours, forest depth),
# and add features about the opponent’s defence.